### ⚠️ Patched for Local Execution
This notebook was originally designed for Google Colab. It has been automatically patched:
- Google Colab-specific imports and `drive.mount()` calls have been commented out
- Colab file paths (`/content/drive/...`) have been replaced with relative paths (`./`)
- `!pip install` commands have been commented out (install packages in your venv instead)

**To run locally:** activate your Python virtual environment first, then run this notebook in VS Code or Jupyter.

In [ ]:
# [PATCHED] from google.colab import drive
# [PATCHED] drive.mount('./')

import pandas as pd
import os

working_folder='./ Drive/TransformersCode/02-ECommerce/CustomerFeedbackAnalysis/'

csv_file_path = os.path.join(working_folder, 'ProductsReviews.csv')
df = pd.read_csv(csv_file_path)

df.head()

In [ ]:
df = df[["reviews.text","sentiment"]]

In [ ]:
def map_sentiment_to_class(sentiment):

    if sentiment=="Negative":
        return 0

    elif sentiment=="Neutral":
        return 1

    elif sentiment=="Positive":
        return 2

df.loc[:, "Class"] = df["sentiment"].apply(map_sentiment_to_class)

In [ ]:
total_rows=300

class_rows = int(total_rows/3)

df_class_0 = df[df['Class'] == 0]
df_class_1 = df[df['Class'] == 1]
df_class_2 = df[df['Class'] == 2]

df_sample_0 = df_class_0.sample(n=min(class_rows, len(df_class_0)), random_state=42)
df_sample_1 = df_class_1.sample(n=min(class_rows, len(df_class_1)), random_state=42)
df_sample_2 = df_class_2.sample(n=min(class_rows, len(df_class_2)), random_state=42)

df_final = pd.concat([df_sample_0, df_sample_1, df_sample_2])

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name_pretrained = "cardiffnlp/twitter-roberta-base-sentiment"

tokenizer = AutoTokenizer.from_pretrained(model_name_pretrained)

model = AutoModelForSequenceClassification.from_pretrained(model_name_pretrained)

In [ ]:
sentence = "I hate this movie"

tokenized_input = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True, max_length=512)
print(tokenized_input)

output = model(**tokenized_input)
print(output)

logits = output.logits

predicted_class = torch.argmax(logits).item()
print(predicted_class)

In [ ]:
sentence = "I am student"

tokenized_input = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True, max_length=512)

output = model(**tokenized_input)

logits = output.logits

predicted_class = torch.argmax(logits).item()
print(predicted_class)

In [ ]:
sentence = "The movie is excellent"

tokenized_input = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True, max_length=512)

output = model(**tokenized_input)

logits = output.logits

predicted_class = torch.argmax(logits).item()
print(predicted_class)

In [ ]:
def get_accuracy_pt(model, tokenizer, df_param):

    df = df_param.copy()

    df["PredClass"] = 0

    for index, row in df.iterrows():

        sentence = row["reviews.text"]

        tokenized_input = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True, max_length=512)

        output = model(**tokenized_input)

        logits = output.logits

        predicted_class = torch.argmax(logits).item()

        df.at[index, "PredClass"] = predicted_class

    correct_predictions = (df["Class"] == df["PredClass"]).sum()

    total_predictions = len(df)

    accuracy = correct_predictions / total_predictions
    return round(100*accuracy,2)

In [ ]:
acc = get_accuracy_pt(model, tokenizer, df_final)

In [ ]:
acc